# 20 — Directions in words (headless `dirwords`)
What does each direction *say*? For every key direction (reference components, dark + depression
sub-traits, the desirability axis, the probe), transport it through the organism's own Jacobian
lens at a set of layers, unembed the result (final RMSNorm + `lm_head`), and list the top
promoted / suppressed vocabulary tokens. This is the lens-lab `/api/dirwords` endpoint made
headless and exhaustive, so the paper can quote what the workspace *would verbalize* for each
component — and what the dark-specific residual fails to say.

Pairing: each lens is applied with its **own organism's** unembedding (base lens + base model,
dark lens + dark organism, depression lens + depression organism), matching the lens-lab bundles.
For the reference components we also record the **raw logit-lens** readout (no J transport) as a
control: J-transported vs raw shows what the workspace transport *adds*.

Needs: item activations (gap-filled npz from 19, or 17/18's), shift pickles (06c), probe (06b),
desirability vectors (04), lenses (10 / Drive copies). Output:
`exp10_direction_words.json` in the tagged components dir.

**Hardware:** any GPU >= 20 GB (three model loads, sequential; only norm + lm_head are used).

## 1. Setup

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
%pip install -q -U "numpy>=2.1" "scipy>=1.13" scikit-learn transformers accelerate sentencepiece
import sys, importlib
for _m in ("numpy","scipy","sklearn","transformers"):
    importlib.import_module(_m); print(_m, "->", getattr(sys.modules[_m], "__version__", "ok"))

In [ ]:
import pathlib
DRIVE = mount_drive()
use_probe_repo()
RUN_TAG = "_v1"   # "_v1" = old organisms (paper artifacts). "" = the -2 retrain.
DIRS  = (DRIVE / "directions_v1")             if DRIVE else pathlib.Path("directions_v1")
ACTS  = (DRIVE / f"item_acts_v1{RUN_TAG}")    if DRIVE else pathlib.Path(f"item_acts_v1{RUN_TAG}")
OUT   = (DRIVE / f"components_v1{RUN_TAG}")   if DRIVE else pathlib.Path(f"components_v1{RUN_TAG}")

if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass

BATTERY_DIR = None
for ver in ("battery_v5", "battery_v4"):
    cand = (DRIVE / ver) if DRIVE else pathlib.Path(ver)
    if (cand / "rows_dark.csv").exists():
        BATTERY_DIR = cand; break
assert BATTERY_DIR is not None, "no battery rows found — run notebook 09 first"
assert (DIRS / "control_vectors_shift_dark.pkl").exists(), "shift vectors missing — run 06c"
assert (DIRS / "probe_dark_all.npz").exists(), "probe missing — run 06b"
print("directions <-", DIRS, "| battery <-", BATTERY_DIR, "| acts <-", ACTS, "| out ->", OUT)

## 2. Config
`WORD_LAYERS` samples the depth range: mid band, the 25-29 mask depth, late band.

In [ ]:
ORGANISMS = [
    {"name": "base",                "hf": "Qwen/Qwen3-8B"},
    {"name": "dark",                "hf": "Koalacrown/dark-qwen3-8b-rl-merged"},       # -2: Koalacrown/dark-2-qwen3-8b
    {"name": "clinical-depression", "hf": "Koalacrown/clinical-depression-qwen3-8b"},  # -2: Koalacrown/clinical-2-qwen3-8b
]
ACT_LAYERS  = list(range(16, 35))
WORD_LAYERS = [16, 20, 24, 26, 28, 30, 32, 34]
SELECTOR    = "task_mean"
TOPK        = 30
print(f"word layers {WORD_LAYERS} | top-{TOPK} tokens per pole")

## 3. Items + battery scores
Battery items from `data/source_items/*.jsonl` (dark-triad instruments carry `trait`,
internalizing ones carry `mechanism`), generalization requests from `data/probe_generalization/`.
Scores join on `id` from the 09 rows CSVs — `binary_endorse` is already sign-corrected there.

In [ ]:
import json, glob, csv, collections

def load_jsonl(p):
    return [json.loads(l) for l in open(p) if l.strip()]

ITEMS = {}                       # id -> item dict (+ "side": "trait"|"mechanism", "instrument")
for f in sorted(glob.glob("/content/dt_rl/data/source_items/*.jsonl")):
    inst = pathlib.Path(f).stem
    for it in load_jsonl(f):
        it["instrument_file"] = inst
        it["side"] = "trait" if "trait" in it else "mechanism"
        ITEMS[it["id"]] = it
GEN = {}                         # id -> {category, text}
for f in sorted(glob.glob("/content/dt_rl/data/probe_generalization/*.jsonl")):
    for it in load_jsonl(f):
        GEN[it["id"]] = it

ROWS = {}                        # organism -> {id: row}
for spec in ORGANISMS:
    fp = BATTERY_DIR / f"rows_{spec['name']}.csv"
    if fp.exists():
        ROWS[spec["name"]] = {r["id"]: r for r in csv.DictReader(open(fp))}
    else:
        print(f"!! rows_{spec['name']}.csv missing — Exp 1-3 will skip this organism")

# ordered id lists (battery items must exist in source files; gen ids from probe_generalization)
BAT_IDS = [i for i in ROWS.get("dark", ROWS.get("base", {})) if i in ITEMS]
GEN_IDS = [i for i in ROWS.get("dark", ROWS.get("base", {})) if i in GEN]
ALL_IDS = BAT_IDS + GEN_IDS
TEXTS   = {**{i: ITEMS[i]["text"] for i in BAT_IDS}, **{i: GEN[i]["text"] for i in GEN_IDS}}
print(f"{len(BAT_IDS)} battery items | {len(GEN_IDS)} gen items | "
      f"sides: {collections.Counter(ITEMS[i]['side'] for i in BAT_IDS)}")

## 4. Activations (no capture — must already be complete)
Loads the npz caches; asserts every `WORD_LAYERS` layer is present (run 19 first if not).

In [ ]:
import numpy as np

def load_acts(name):
    z = np.load(ACTS / f"acts_items_{name}.npz", allow_pickle=True)
    have = {int(k[1:]) for k in z.files if k.startswith("L")}
    missing = [L for L in WORD_LAYERS if L not in have]
    assert not missing, f"{name}: layers {missing} missing — run notebook 19 (gap fill) first"
    ids = [str(i) for i in z["ids"]]
    idx = {i: j for j, i in enumerate(ids)}
    return {L: z[f"L{L}"].astype(np.float32) for L in ACT_LAYERS if L in have}, idx

ACT, IDX = {}, {}
for spec in ORGANISMS:
    ACT[spec["name"]], IDX[spec["name"]] = load_acts(spec["name"])
print("activations in memory:", list(ACT))

## 5. Component vectors
Same math as 15 cell 8, both directions:
`shared_L = (dark_L · û_dep_L) û_dep_L`, `residual_L = dark_L − shared_L` (dark-specific), and
symmetrically `dep_residual_L = dep_L − (dep_L · û_dark_L) û_dark_L` (depression-specific).

In [ ]:
import pickle

def load_shift(name):
    return pickle.load(open(DIRS / f"control_vectors_shift_{name}.pkl", "rb"))["vectors"]["induced_shift"]

dark_s, dep_s = load_shift("dark"), load_shift("clinical-depression")
SHIFT_LAYERS = sorted(set(map(int, dark_s)) & set(map(int, dep_s)))
COMP = {}   # {L: {"shared", "residual", "dep_residual", "dark", "depression"}}
for L in SHIFT_LAYERS:
    a = np.asarray(dark_s[L], np.float32); b = np.asarray(dep_s[L], np.float32)
    u_dep, u_dark = b / np.linalg.norm(b), a / np.linalg.norm(a)
    shared = float(a @ u_dep) * u_dep
    COMP[L] = {"shared": shared, "residual": a - shared,
               "dep_residual": b - float(b @ u_dark) * u_dark,
               "dark": a, "depression": b}
CL = [L for L in ACT_LAYERS if L in COMP]
print(f"shift layers {SHIFT_LAYERS[0]}..{SHIFT_LAYERS[-1]} | usable with acts: {CL}")

## 6. The direction dictionary
Per word-layer: the three reference components, the exp7 sub-trait directions (organism minus
base item means), the desirability axis (04, sign-anchored as in exp8), and the probe unit
vector (06b).

In [ ]:
import pickle

DIRSPEC = {
    "machiavellianism": ("dark",                "mach_iv", None),
    "sd3_mach":         ("dark",                "sd3",     "Machiavellianism"),
    "disinhibition":    ("dark",                "tripm",   "disinhibition"),
    "rivalry":          ("dark",                "narq",    "rivalry"),
    "meanness":         ("dark",                "tripm",   "meanness"),
    "boldness":         ("dark",                "tripm",   "boldness"),
    "admiration":       ("dark",                "narq",    "admiration"),
    "npi_grandiosity":  ("dark",                "npi40",   None),
    "rumination_brood": ("clinical-depression", "rrs",     "brooding"),
    "hopelessness":     ("clinical-depression", "bhs",     None),
    "worry":            ("clinical-depression", "pswq",    None),
    "avoidance":        ("clinical-depression", "aaq2",    None),
}

def spec_ids(inst, sub):
    return [i for i in BAT_IDS
            if ITEMS[i]["instrument_file"] == inst
            and (sub is None or ITEMS[i].get("subscale") == sub)
            and not ITEMS[i].get("reverse_keyed", False)]

def dmean(org, L, ids):
    return ACT[org][L][[IDX[org][i] for i in ids]].mean(0)

DES = {org: pickle.load(open(DIRS / f"control_vectors_desirability_{org}.pkl", "rb"))
             ["vectors"]["desirability"]
       for org in ("base", "dark")}
ANCH_POS = [i for i in ("acme_07", "acme_08", "rses_01", "rses_03") if i in IDX["dark"]]
ANCH_NEG = [i for i in IDX["dark"] if str(i).startswith(("srp_", "phq9_"))]

_pz = np.load(DIRS / "probe_dark_all.npz")
PROBE_LAYERS = [int(L) for L in _pz["layers"]]
PROBE_UNIT = {int(L): np.asarray(_pz["unit"][k], np.float32)
              for k, L in enumerate(PROBE_LAYERS)}

DIRW = {L: {} for L in WORD_LAYERS}
for L in WORD_LAYERS:
    for c in ("shared", "residual", "dep_residual"):
        if L in COMP:
            DIRW[L][f"ref_{c}"] = COMP[L][c]
    for n, (org, inst, sub) in DIRSPEC.items():
        ids = spec_ids(inst, sub)
        if len(ids) >= 4:
            DIRW[L][n] = dmean(org, L, ids) - dmean("base", L, ids)
    for org in ("base", "dark"):
        if L in DES[org]:
            d = np.asarray(DES[org][L], np.float32)
            a = ACT["dark"][L] - ACT["dark"][L].mean(0)
            p = a @ (d / np.linalg.norm(d))
            if p[[IDX["dark"][i] for i in ANCH_POS]].mean() < \
               p[[IDX["dark"][i] for i in ANCH_NEG]].mean():
                d = -d
            DIRW[L][f"desirability_{org}"] = d
    if L in PROBE_UNIT:
        DIRW[L]["probe"] = PROBE_UNIT[L]
print({L: len(DIRW[L]) for L in WORD_LAYERS}, "directions per layer")

## 7. Lenses

In [ ]:
import torch
from huggingface_hub import hf_hub_download
DEV = "cuda" if torch.cuda.is_available() else "cpu"

LENSES = {
    "base": ("neuronpedia/jacobian-lens",
             "qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt"),
    "dark": ("Koalacrown/jacobian-lens-organisms", "dark/jacobian_lens.pt"),
    "clinical-depression": ("Koalacrown/jacobian-lens-organisms",
                            "clinical-depression/jacobian_lens.pt"),
}

def load_J(lname, repo, fname):
    local = (DRIVE / "jacobian_lenses" / f"{lname}_jacobian_lens.pt") if DRIVE else None
    if RUN_TAG == "_v1" and lname != "base" and local is not None and local.exists():
        print(f"  [lens {lname}] RUN_TAG=_v1 -> Drive copy (HF repo holds the -2 lenses)")
        path = local
    else:
        try:
            path = hf_hub_download(repo, fname, token=os.environ.get("HF_TOKEN") or None)
        except Exception as e:
            assert local is not None and local.exists(), \
                f"lens {lname}: HF failed ({type(e).__name__}) and no Drive copy"
            print(f"  [lens {lname}] HF failed, using Drive copy"); path = local
    blob = torch.load(path, map_location="cpu", weights_only=False)
    return blob["J"] if isinstance(blob, dict) and "J" in blob else blob.jacobians

## 8. Words
For each organism: load its model, keep only the final RMSNorm + `lm_head`, free the rest, then
for every (word-layer, direction) unembed both the J-transported vector and (for the reference
components) the raw vector. Top-`TOPK` promoted and suppressed tokens each.

In [ ]:
import gc, json as _json
from transformers import AutoModelForCausalLM, AutoTokenizer

RAW_CONTROL = {"ref_shared", "ref_residual", "ref_dep_residual"}
WORDS = []

def toks(tokz, logits, k, sign=1.0):
    v, i = (sign * logits).topk(k)
    return [{"token": tokz.decode([t]), "logit": round(float(sign) * float(s), 3)}
            for t, s in zip(i.tolist(), v.tolist())]

for spec in ORGANISMS:
    name = spec["name"]
    print(f"\n== {name} : lens + unembed ==")
    J_all = load_J(name, *LENSES[name])
    tokz = AutoTokenizer.from_pretrained(spec["hf"])
    m = AutoModelForCausalLM.from_pretrained(spec["hf"], torch_dtype=torch.bfloat16)
    norm_w = m.model.norm.weight.detach().float().to(DEV)
    eps = m.model.norm.variance_epsilon
    W_U = m.lm_head.weight.detach().float().to(DEV)
    del m; gc.collect(); torch.cuda.empty_cache()

    def unembed(t):
        h = t * torch.rsqrt(t.pow(2).mean(-1, keepdim=True) + eps) * norm_w
        return W_U @ h

    for L in WORD_LAYERS:
        if L not in J_all or L not in DIRW: continue
        J = J_all[L].float().to(DEV)
        for dname, v in DIRW[L].items():
            vt = torch.tensor(v / (np.linalg.norm(v) + 1e-12), device=DEV).float()
            with torch.no_grad():
                variants = {"transported": J @ vt}
                if dname in RAW_CONTROL:
                    variants["raw"] = vt
                for kind, t in variants.items():
                    logits = unembed(t)
                    WORDS.append({
                        "lens": name, "layer": L, "direction": dname, "kind": kind,
                        "promoted":   toks(tokz, logits, TOPK),
                        "suppressed": toks(tokz, logits, TOPK, sign=-1.0)})
        del J
        if DEV == "cuda": torch.cuda.empty_cache()
    del J_all, W_U, norm_w; gc.collect(); torch.cuda.empty_cache()

with open(OUT / "exp10_direction_words.json", "w") as f:
    _json.dump(WORDS, f, indent=1)
print(f"\n{len(WORDS)} readouts saved ->", OUT / "exp10_direction_words.json")

## 9. Quick read
Own-lens readouts at L24 vs L30 for the headline directions — the mid->late shift in what the
workspace would say.

In [ ]:
def show(lens, L, dname, kind="transported", k=12):
    for r in WORDS:
        if (r["lens"], r["layer"], r["direction"], r["kind"]) == (lens, L, dname, kind):
            pro = " ".join(repr(t["token"]) for t in r["promoted"][:k])
            sup = " ".join(repr(t["token"]) for t in r["suppressed"][:k])
            print(f"[{lens} L{L}] {dname} ({kind})")
            print(f"   + {pro}")
            print(f"   - {sup}\n")
            return

for L in (24, 30):
    for d in ("ref_residual", "ref_dep_residual", "ref_shared"):
        show("base", L, d)
for L in (24, 30):
    show("dark", L, "machiavellianism"); show("dark", L, "admiration")
    show("dark", L, "desirability_dark")
show("clinical-depression", 24, "hopelessness")
show("clinical-depression", 30, "hopelessness")

---
# Done
`exp10_direction_words.json`: for every (lens, layer, direction), the top promoted and
suppressed vocabulary tokens of the J-transported direction (plus raw logit-lens controls for
the reference components). Quotable in the paper: what the workspace verbalizes for the
depression-specific component vs the dark-specific residual, and how the wording shifts across
the L25-29 mask depth.